In [1]:
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
openai_client = OpenAI()

In [2]:
from openai import OpenAI
import os

openai_client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

In [3]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

openai_client = OpenAI()

def llm(prompt):
    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=prompt,
    )
    return response.output_text

In [4]:
llm("Hey, what's up?")

'Hey! Not much—just here and ready to help. What’s on your mind?'

In [5]:
question = "I just discovered the course. Can I join now?"
answer = llm(question)
print(answer)

It depends on the course and how it’s set up.

If you just discovered it, you may still be able to join if:
- enrollment is still open,
- there’s room left,
- and you haven’t missed any hard deadlines.

If you want, I can help you figure out the best wording to ask, for example:

“Hi, I just discovered the course. Is it still possible to join at this point?”

If this is for a real course, tell me the course name and platform/school, and I can help you check what to do next or draft a message.


In [6]:
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

In [7]:
prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
"""

In [8]:
answer = llm(prompt)
print(answer)

Yes, you can still join now. If you want to receive a certificate, make sure to submit your project while submissions are still open.


In [9]:
def rag(question):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt)

In [10]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

In [11]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1275

In [12]:
documents[0]

{'id': '74eb249bbf',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

In [13]:
documents[1]

{'id': '977bf7786c',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
 'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."}

In [14]:
documents[600]

{'id': '80e735cca8',
 'course': 'machine-learning-zoomcamp',
 'section': 'Module 3. Machine Learning for Classification',
 'question': 'What is the best way to handle missing values in the dataset before training a regression model?',
 'answer': 'You can handle missing values by:\n\n- Imputing the missing values with the mean, median, or mode.\n- Using algorithms that support missing values inherently (e.g., some tree-based methods).\n- Removing rows or columns with missing data, depending on the extent of missingness.\n- Utilizing feature engineering to derive new features from incomplete data.'}

In [15]:
from minsearch import Index

index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)

In [16]:
question = "I just discovered the course. Can I join now?"

search_results = index.search(
    question,
    boost_dict={"question": 2.0, "section": 0.5},
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

In [17]:
[doc["question"] for doc in search_results]

['I just discovered the course. Can I still join?',
 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
 'When will the course be offered next?',
 'I missed the first homework - can I still get a certificate?']

In [18]:
results = index.search(
    question,
    num_results=5,
    boost_dict={"question": 2.0, "section": 0.5}
)

In [19]:
results = index.search(
    question,
    num_results=5,
    filter_dict={"course": "mlops-zoomcamp"}
)

In [20]:
def search(question, course="llm-zoomcamp"):
    boost_dict = {"question": 2.0, "section": 0.5}
    filter_dict = {"course": course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )

In [21]:
search_results = search(question)

In [22]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

In [23]:
USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

In [24]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()

In [25]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

In [26]:
prompt = build_prompt(question, search_results)

print(prompt)

Question:
I just discovered the course. Can I join now?

Context:
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project

In [27]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=prompt
)

In [28]:
response.output[0]

ResponseOutputMessage(id='msg_01ff67f5f81f4a30006a29727d7ac08191893e832947659b91', content=[ResponseOutputText(annotations=[], text='Yes — you can still join now.\n\nIf you want a certificate, make sure to submit your project while submissions are still open. You can also start learning and submitting homework as long as the forms are open.', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase='final_answer')

In [29]:
response.output_text

'Yes — you can still join now.\n\nIf you want a certificate, make sure to submit your project while submissions are still open. You can also start learning and submitting homework as long as the forms are open.'

In [30]:
response.usage

ResponseUsage(input_tokens=334, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=46, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=380)

In [31]:
input_price = 0.75 / 1_000_000
output_price = 4.50 / 1_000_000

cost = (
    response.usage.input_tokens * input_price +
    response.usage.output_tokens * output_price
)

cost

0.0004575

In [32]:
message_history = [
    {"role": "developer", "content": INSTRUCTIONS},
    {"role": "user", "content": prompt}
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=message_history
)

In [33]:
def llm(instructions, user_prompt, model="gpt-5.4-mini"):
    message_history = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.responses.create(
        model=model,
        input=message_history
    )

    return response.output_text

In [34]:
def rag(query, model="gpt-5.4-mini"):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model=model)
    return answer

In [35]:
answer = rag("I just discovered the course. Can I join now?")
print(answer)

Yes, but if you want to receive a certificate, you need to submit your project while submissions are still being accepted.


In [36]:
rag("How do I get a certificate?")

'You can get a certificate only if you finish the course with a **live cohort**.\n\nA certificate is **not** awarded for the self-paced mode, because you need to **peer-review 3 capstone projects** after submitting your own project, and that only happens while the course is running.\n\nIf you want, I can also tell you whether **missing homework** affects certificate eligibility.'

In [37]:
from rag_helper import INSTRUCTIONS, PROMPT_TEMPLATE

In [38]:
class RAGBase:

    def __init__(
        self,
        index,
        llm_client,
        instructions=INSTRUCTIONS,
        prompt_template=PROMPT_TEMPLATE,
        course="llm-zoomcamp",
        model="gpt-5.4-mini"
    ):
        self.index = index
        self.llm_client = llm_client
        self.instructions = instructions
        self.course = course
        self.prompt_template = prompt_template
        self.model = model

In [39]:
class RAGBase:

    def __init__(
        self,
        index,
        llm_client,
        instructions=INSTRUCTIONS,
        prompt_template=PROMPT_TEMPLATE,
        course="llm-zoomcamp",
        model="gpt-5.4-mini"
    ):
        self.index = index
        self.llm_client = llm_client
        self.instructions = instructions
        self.course = course
        self.prompt_template = prompt_template
        self.model = model

In [40]:
    def search(self, query, num_results=5):
        boost_dict = {"question": 3.0, "section": 0.5}
        filter_dict = {"course": self.course}

        return self.index.search(
            query,
            num_results=num_results,
            boost_dict=boost_dict,
            filter_dict=filter_dict
        )

In [41]:
    def build_context(self, search_results):
        lines = []

        for doc in search_results:
            lines.append(doc["section"])
            lines.append("Q: " + doc["question"])
            lines.append("A: " + doc["answer"])
            lines.append("")

        return "\n".join(lines).strip()

    def build_prompt(self, query, search_results):
        context = self.build_context(search_results)
        return self.prompt_template.format(
            question=query, context=context
        )

In [42]:
    def llm(self, prompt):
        input_messages = [
            {"role": "developer", "content": self.instructions},
            {"role": "user", "content": prompt}
        ]

        response = self.llm_client.responses.create(
            model=self.model,
            input=input_messages
        )

        return response.output_text


In [43]:
    def rag(self, query):
        search_results = self.search(query)
        prompt = self.build_prompt(query, search_results)
        answer = self.llm(prompt)
        return answer

In [44]:
from dotenv import load_dotenv
load_dotenv(".env")

from ingest import load_faq_data, build_index
from rag_helper import RAGBase
from openai import OpenAI

documents = load_faq_data()
index = build_index(documents)

openai_client = OpenAI()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
)

answer = assistant.rag("I just discovered the course. Can I join now?")
print(answer)

Yes, you can still join. If you want a certificate, make sure to submit your project while submissions are still open.


In [45]:
custom_instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=custom_instructions,
)

In [46]:
assistant.rag("How do I get a certificate?")
assistant.rag("Can I still join the course after it started?")

'Yes, you can still join the course after it started. If you want to receive a certificate, you need to submit your project while submissions are still being accepted.'